In [2]:
"""
Hardware resource estimate for the deterministic decoder circuit (build_dctc)
on ibm_fez, for m = 1, 2, 3.

Reports, for each m, both PRE-transpile and POST-transpile metrics:
    - total gate count
    - two-qubit gate count
    - depth
    - two-qubit depth
    - count_ops breakdown

Fill in IBM_TOKEN and IBM_CRN below before running.
"""

from math import pi
import pandas as pd

from qiskit import transpile
from qiskit_ibm_runtime import QiskitRuntimeService

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from math import pi

def build_dctc(n: int, msg_params=None):
    """
    Step 1: Prepare message states on M[i] and swap them into C[i].
    (No entanglement, no scrambling/decoding, no measurements.)

    Args
    ----
    n : int
        Number of logical CTC qubits.
    msg_params : list[tuple[float, float, float]] | None
        Per-qubit single-qubit U gate parameters (theta, phi, lam) for M[i].
        If None, uses [(0,0,0)] for all i.

    Returns
    -------
    qc : QuantumCircuit
        Circuit containing only Step 1, with full 7n registers allocated
        in the order: C, E, R, G, M, A, Y, crA, crY, crC.
    """
    if msg_params is None:
      print("Please input the message parameters theta, phi, lambda")
    #     msg_params = [(0.0, 0.0, 0.0)] * n
    # assert len(msg_params) == n, "msg_params must have length n"

    # --- Allocate all registers now (to keep wire order consistent for later steps) ---
    C = QuantumRegister(n, 'C')   # CTC
    E = QuantumRegister(n, 'E')   # Early radiation
    R = QuantumRegister(n, 'R')   # Recent radiation
    G = QuantumRegister(n, 'G')   # Grover/projection ancilla
    M = QuantumRegister(n, 'M')   # Message
    A = QuantumRegister(n, 'A')   # Decoder ancilla
    Y = QuantumRegister(n, 'Y')   # Output
    crR = ClassicalRegister(n, 'crR')
    crG = ClassicalRegister(n, 'crG')
    crC = ClassicalRegister(n, 'crC')

    qc = QuantumCircuit(C, E, R, G, M, A, Y, crR, crG, crC)

    # --- Step 1a: Prepare messages on M[i] ---
    for i, (theta, phi, lam) in enumerate(msg_params):
        qc.u(theta, phi, lam, M[i])

    # --- Step 1b: Load messages into the CTC qubits ---
    for i in range(n):
        qc.swap(C[i], M[i])

    qc.barrier()  # delimiter: messages loaded into C

    # ---- EPR Pairs -----

    for i in range(n):
        # (E[i], M[i])
        qc.h(E[i])
        qc.cx(E[i], M[i])

        # (R[i], G[i])
        qc.h(R[i])
        qc.cx(R[i], G[i])

        # (A[i], Y[i])
        qc.h(A[i])
        qc.cx(A[i], Y[i])

    qc.barrier()

        # --- Scrambling Unitary (per qubit) ---
    for i in range(n):
        qc.cz(C[i], R[i])
        qc.cz(E[i], R[i])
        qc.cz(C[i], E[i])
        qc.h(C[i])
        qc.h(E[i])
        qc.h(R[i])
        qc.cz(C[i], R[i])
        qc.cz(C[i], E[i])
        qc.cz(E[i], R[i])
        qc.barrier()

    # --- Unitary Conjugate (Decoder) (per qubit) ---
    for i in range(n):
        qc.cz(A[i], G[i])
        qc.cz(M[i], A[i])
        qc.cz(G[i], M[i])
        qc.h(A[i])
        qc.h(M[i])
        qc.h(G[i])
        qc.cz(A[i], G[i])
        qc.cz(G[i], M[i])
        qc.cz(M[i], A[i])
        qc.barrier()

    # ---- Grover operator (R,G) ----

    for i in range(n):
      qc.rz(pi, R[i])
      qc.rx(pi, R[i])
      qc.rx(pi, G[i])
      qc.swap(R[i], G[i])
      qc.rz(pi, R[i])
      qc.barrier()


    # --- Unitary Transpose (Decoder) (per qubit) ---
    for i in range(n):
      qc.cz(A[i], G[i])
      qc.cz(M[i], A[i])
      qc.cz(G[i], M[i])
      qc.h(A[i])
      qc.h(M[i])
      qc.h(G[i])
      qc.cz(A[i], G[i])
      qc.cz(G[i], M[i])
      qc.cz(M[i], A[i])
      qc.barrier()

     # ---- Grover operator (A,Y) ----

    for i in range(n):
      qc.rz(pi, A[i])
      qc.rx(pi, A[i])
      qc.rx(pi, Y[i])
      qc.swap(A[i], Y[i])
      qc.rz(pi, A[i])
      qc.barrier()


   # --- Unitary Conjugate (Decoder) (per qubit) ---
    for i in range(n):
      qc.cz(A[i], G[i])
      qc.cz(M[i], A[i])
      qc.cz(G[i], M[i])
      qc.h(A[i])
      qc.h(M[i])
      qc.h(G[i])
      qc.cz(A[i], G[i])
      qc.cz(G[i], M[i])
      qc.cz(M[i], A[i])
      qc.barrier()


    for i in range(n):
      qc.rz(pi, R[i])
      qc.rx(pi, R[i])
      qc.rx(pi, G[i])
      qc.swap(R[i], G[i])
      qc.rz(pi, R[i])
      qc.barrier()



    # --- Bell Projection per qubit ---
    for i in range(n):
        qc.cx(R[i], G[i])
        qc.h(R[i])
        qc.measure(R[i], crR[i])
        qc.measure(G[i], crG[i])
        qc.barrier()

    # --- Getting back input message into C  ---


    for i in range(n):
        qc.swap(C[i], Y[i])

    # for i, (theta, phi, lam) in enumerate(msg_params):
    #     qc.u(theta, phi, lam, C[i]).inverse()

    # for i in range(n):
    #     qc.measure(C[i], crC[i])


    return qc


# ---------------------------------------------------------------------------
# Credentials (fill in)
# ---------------------------------------------------------------------------
IBM_TOKEN = "WK7pTqZV44cncanG_5HxDG7am1plhkD6TSaQd57Sc-yI"    # <-- paste your IBM Quantum API token here
IBM_CRN   = "crn:v1:bluemix:public:quantum-computing:us-east:a/c869b082184242d5b049e69e9ca64c47:7d7a0de3-23aa-487f-9c9a-dbc2f8e36ed3::"    # <-- paste your instance CRN here
BACKEND_NAME = "ibm_fez"

# Transpilation settings
OPT_LEVEL = 1     # set to 3 if you want aggressive optimization for "as-submitted" numbers
SEED = 42         # fixed seed so numbers are reproducible across runs

# Message parameters per qubit (theta, phi, lambda); theta=pi/3 matches project convention
def default_msg_params(n):
    return [(pi/3, 0.0, 0.0)] * n

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def two_qubit_ops(qc):
    """Count gates acting on 2+ qubits, excluding barriers/measures/resets."""
    skip = {"barrier", "measure", "reset", "delay", "snapshot"}
    return sum(1 for instr in qc.data
               if instr.operation.name not in skip and len(instr.qubits) >= 2)

def two_qubit_depth(qc):
    """Depth restricted to multi-qubit gates."""
    return qc.depth(lambda instr: len(instr.qubits) >= 2
                    and instr.operation.name not in {"barrier", "measure", "reset", "delay"})

def total_gate_count(qc):
    skip = {"barrier", "measure", "reset", "delay", "snapshot"}
    return sum(1 for instr in qc.data if instr.operation.name not in skip)

def summarize(qc, label):
    return {
        "label": label,
        "num_qubits": qc.num_qubits,
        "total_gates": total_gate_count(qc),
        "two_qubit_gates": two_qubit_ops(qc),
        "depth": qc.depth(),
        "two_qubit_depth": two_qubit_depth(qc),
        "ops": dict(qc.count_ops()),
    }

# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main():
    if not IBM_TOKEN or not IBM_CRN:
        raise RuntimeError("Fill in IBM_TOKEN and IBM_CRN at the top of the script.")

    service = QiskitRuntimeService(
        channel="ibm_quantum_platform",
        token=IBM_TOKEN,
        instance=IBM_CRN,
    )
    backend = service.backend(BACKEND_NAME)
    print(f"Backend: {backend.name}")
    print(f"  num_qubits: {backend.num_qubits}")
    print(f"  basis_gates: {backend.configuration().basis_gates}")
    print()

    rows = []
    for m in (1, 2, 3):
        print(f"=== m = {m} ===")
        qc = build_dctc(m, msg_params=default_msg_params(m))

        pre = summarize(qc, f"m={m} pre-transpile")
        print(f"  pre  : qubits={pre['num_qubits']}, total={pre['total_gates']}, "
              f"2Q={pre['two_qubit_gates']}, depth={pre['depth']}, "
              f"2Q-depth={pre['two_qubit_depth']}")
        print(f"         ops: {pre['ops']}")

        qc_t = transpile(qc, backend=backend, optimization_level=OPT_LEVEL, seed_transpiler=SEED)

        post = summarize(qc_t, f"m={m} post-transpile (ibm_fez, opt={OPT_LEVEL})")
        print(f"  post : qubits={post['num_qubits']}, total={post['total_gates']}, "
              f"2Q={post['two_qubit_gates']}, depth={post['depth']}, "
              f"2Q-depth={post['two_qubit_depth']}")
        print(f"         ops: {post['ops']}")
        print()

        rows.append({"m": m, "stage": "pre",  **{k: v for k, v in pre.items()  if k != "ops"}})
        rows.append({"m": m, "stage": "post", **{k: v for k, v in post.items() if k != "ops"}})

    df = pd.DataFrame(rows)
    print("\n=== Summary ===")
    print(df.to_string(index=False))
    df.to_csv("resource_estimate_fez.csv", index=False)
    print("\nSaved: resource_estimate_fez.csv")

if __name__ == "__main__":
    main()

qiskit_runtime_service._discover_account:WARNING:2026-06-16 15:35:27,969: Loading account with the given token. A saved account will not be used.


Backend: ibm_fez
  num_qubits: 156
  basis_gates: ['cz', 'id', 'rz', 'sx', 'x']

=== m = 1 ===
  pre  : qubits=7, total=62, 2Q=33, depth=48, 2Q-depth=31
         ops: {'cz': 24, 'h': 16, 'barrier': 10, 'rz': 6, 'rx': 6, 'swap': 5, 'cx': 4, 'measure': 2, 'u': 1}
  post : qubits=156, total=340, 2Q=106, depth=205, 2Q-depth=97
         ops: {'sx': 173, 'cz': 106, 'rz': 61, 'barrier': 10, 'measure': 2}

=== m = 2 ===
  pre  : qubits=14, total=124, 2Q=66, depth=91, 2Q-depth=59
         ops: {'cz': 48, 'h': 32, 'barrier': 18, 'rz': 12, 'rx': 12, 'swap': 10, 'cx': 8, 'measure': 4, 'u': 2}
  post : qubits=156, total=671, 2Q=209, depth=377, 2Q-depth=182
         ops: {'sx': 340, 'cz': 209, 'rz': 122, 'barrier': 18, 'measure': 4}

=== m = 3 ===
  pre  : qubits=21, total=186, 2Q=99, depth=134, 2Q-depth=87
         ops: {'cz': 72, 'h': 48, 'barrier': 26, 'rz': 18, 'rx': 18, 'swap': 15, 'cx': 12, 'measure': 6, 'u': 3}
  post : qubits=156, total=1018, 2Q=318, depth=556, 2Q-depth=271
         ops: {'s